# Naive Bayes

In [2]:
from sklearn.datasets import load_breast_cancer
import pandas as pd 

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (569, 30)
y shape: (569,)


In [3]:
print(X.head())
print(y.value_counts())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0           

### Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114


### Create Gaussian Naive Bayes

In [5]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


### Make predictions

In [6]:
y_train_pred = nb_model.predict(X_train)
y_test_pred = nb_model.predict(X_test)
y_test_proba = nb_model.predict_proba(X_test)

print(y_test_proba[:5])

[[1.00000000e+000 2.64170048e-092]
 [9.68517668e-018 1.00000000e+000]
 [9.99994805e-001 5.19538735e-006]
 [3.39734615e-001 6.60265385e-001]
 [1.00000000e+000 1.39411706e-139]]


In [7]:
print(nb_model.classes_)

[0 1]


In [8]:
y_test_proba[:, 1]

array([2.64170048e-092, 1.00000000e+000, 5.19538735e-006, 6.60265385e-001,
       1.39411706e-139, 1.00000000e+000, 1.00000000e+000, 1.28924043e-052,
       1.40507268e-032, 1.06451408e-110, 1.00000000e+000, 5.80048509e-009,
       1.00000000e+000, 1.67412817e-045, 2.19171328e-014, 9.99999920e-001,
       4.86702752e-001, 9.99999971e-001, 1.00000000e+000, 9.90357067e-001,
       5.06538915e-011, 1.42130136e-007, 1.00000000e+000, 9.99999999e-001,
       1.00000000e+000, 9.98769654e-001, 1.82489207e-099, 1.00000000e+000,
       1.00000000e+000, 1.00000000e+000, 2.73888785e-001, 1.00000000e+000,
       1.00000000e+000, 9.99990839e-001, 8.94592713e-045, 9.99999512e-001,
       1.00000000e+000, 9.99999870e-001, 9.99999657e-001, 4.15237133e-057,
       1.00000000e+000, 1.00000000e+000, 1.00000000e+000, 3.46206506e-133,
       3.43067746e-004, 9.93998059e-001, 1.00000000e+000, 1.00000000e+000,
       9.31108226e-001, 9.61486789e-025, 1.00000000e+000, 9.99999450e-001,
       1.00000000e+000, 9

### Evaluate the model

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

train_f1 = f1_score(y_train, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

print("Train F1:", train_f1)
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1:", test_f1)

Train F1: 0.9532062391681109
Test Accuracy: 0.9385964912280702
Test Precision: 0.9452054794520548
Test Recall: 0.9583333333333334
Test F1: 0.9517241379310345


In [10]:
f1_gap = train_f1 - test_f1

print("F1 Gap:", f1_gap)

F1 Gap: 0.001482101237076372


In [11]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)

print(cm)

[[38  4]
 [ 3 69]]


### Cross-validation

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    nb_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("CV F1 scores:", cv_scores)
print("Mean CV F1:", cv_scores.mean())
print("Std CV F1:", cv_scores.std())

CV F1 scores: [0.94827586 0.95575221 0.94915254 0.94827586 0.94736842]
Mean CV F1: 0.949764979990565
Std CV F1: 0.00304632469526238


# Naive Bayes Evaluation + Model Comparison

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Gaussian NB": GaussianNB(),

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "SVM": SVC(
        kernel="rbf"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss"
    )
}